<div style="line-height: 1.8;">

# **Bakta annotation analysis**

Структурную и первичную функциональную аннотацию генома выполнили с помощью Bakta 1.12.2. Запуск провели 22 октября 2026 года в Docker-контейнере с базой данных `light` версии 6.0. Использовали четыре потока и параметр `--complete`; таксономические сведения передали как `Escherichia`, `coli` и `K-12`.

Полученную аннотацию приняли за основу итогового каталога. Результаты сравнили с сохранённой записью NCBI, не перенося референсные CDS в набор Bakta. Параметры вычисления и версии зависимостей зафиксировали в `results/structural/bakta/ecoli.log`, сведения о версии базы и времени выполнения — в `ecoli.json`.

</div>

In [1]:
from pathlib import Path
from Bio import SeqIO
from collections import Counter
import pandas as pd

<div style="line-height: 1.8;">

### **Структурная аннотация**

Из файла `ecoli.gbff` извлекли геномные признаки и определили их распределение по типам. В полученной аннотации учли 4315 CDS, 88 тРНК, 22 рРНК и 232 записи `ncRNA`. Записи `gene` и связанные с ними CDS рассматривали как разные уровни описания одного локуса, поэтому не суммировали их как независимые гены.

Численность основных типов признаков сопоставили с NCBI. Выявленные различия характеризовали состав аннотаций; оценку соответствия координат выполнили отдельно при сборке итогового выпуска.

</div>

In [2]:
import os

configured_root = os.environ.get("GENOME_ANNOTATION_PROJECT_ROOT")
if configured_root:
    project_dir = Path(configured_root).expanduser().resolve()
else:
    candidates = [candidate for base in (Path.cwd(), *Path.cwd().parents)
                  for candidate in (base, base / "projects/genome_annotation")]
    project_dir = next((p for p in candidates if (p / "data/raw/genome.fna").is_file()), None)
if project_dir is None or not (project_dir / "data/raw/genome.fna").is_file():
    raise FileNotFoundError("Project root not found")

bakta_dir = project_dir / "results/structural/bakta"

genome = SeqIO.read(
    bakta_dir / "ecoli.gbff",
    "genbank"
)

counts = Counter(f.type for f in genome.features)

bakta_counts = pd.DataFrame(
    counts.items(),
    columns=["Feature", "Count"]
).sort_values("Count", ascending=False)

display(bakta_counts)

,Feature,Count
1,gene,4658
2,CDS,4315
4,ncRNA,232
6,tRNA,88
3,regulatory,57
5,rRNA,22
8,repeat_region,2
9,rep_origin,2
0,source,1
7,tmRNA,1


In [3]:
features = ["gene", "CDS", "tRNA", "rRNA", "ncRNA"]

reference = SeqIO.read(project_dir / "data/processed/NC_000913.3.gbk", "genbank")
assert str(reference.seq).upper() == str(genome.seq).upper()
ncbi_counts = Counter(f.type for f in reference.features)

comparison = pd.DataFrame({
    "Feature": features,
    "NCBI": [ncbi_counts[f] for f in features],
    "Bakta": [counts.get(f, 0) for f in features]
})

comparison["Difference"] = (comparison["Bakta"] - comparison["NCBI"])

display(comparison)

,Feature,NCBI,Bakta,Difference
0,gene,4651,4658,7
1,CDS,4318,4315,-3
2,tRNA,86,88,2
3,rRNA,22,22,0
4,ncRNA,108,232,124


<div style="line-height: 1.8;">

### **Функциональная аннотация**

Среди 4315 CDS выделили 234 записи с описанием `hypothetical protein`, что составило 5,42%. Для остальных 4081 записей в аннотации присутствовали иные названия продуктов. Наличие более конкретного названия не рассматривали как экспериментальное подтверждение функции.

Описания продуктов сопоставили с белковым FASTA `ecoli.faa`. Идентификаторы последовательностей использовали при последующем объединении с результатами eggNOG-mapper и InterProScan.

</div>

In [4]:
bakta_cds = [f for f in genome.features if f.type == "CDS"]

products = [f.qualifiers.get("product", ["unknown"])[0] for f in bakta_cds]

hypothetical = sum("hypothetical protein" in p.lower() for p in products)

print("CDS:", len(bakta_cds))
print("Hypothetical proteins:", hypothetical)
print("Other product descriptions:",len(bakta_cds) - hypothetical)
print("Hypothetical fraction:",f"{hypothetical / len(bakta_cds) * 100:.2f}%")

CDS: 4315
Hypothetical proteins: 234
Other product descriptions: 4081
Hypothetical fraction: 5.42%


In [5]:
for feature in bakta_cds[:10]:
    print(
        feature.qualifiers.get("gene", ["-"])[0],
        "|",
        feature.qualifiers.get("product", ["unknown"])[0]
    )

- | hypothetical protein
- | thr operon leader peptide
- | Bifunctional aspartokinase/homoserine dehydrogenase 1
- | Homoserine kinase
- | Threonine synthase
- | hypothetical protein
- | UPF0246 protein GN242_17910
yaaJ | Uncharacterized transporter YaaJ
- | transaldolase B
mogA | Molybdenum cofactor biosynthesis protein MogA


In [6]:
protein_path = (project_dir/ "results/structural/bakta/ecoli.faa")

proteins = list(SeqIO.parse(protein_path, "fasta"))

print("Proteins:", len(proteins))

Proteins: 4315


<div style="line-height: 1.8;">

В белковом FASTA учли 4315 последовательностей, что соответствовало числу CDS в GenBank. При сборке выпуска дополнительно подтвердили согласованность идентификаторов и координат. Полученный набор белков использовали как основу функционального каталога.

</div>